# 10 - Etapa 2: desagregação de cargas (NILM) usando harmônicas + inter-harmônicas (RQ2)

**Pergunta**: com máquina e retificador/diodo operando simultaneamente, é possível separar as duas assinaturas usando só a assinatura espectral combinada? Ver `docs/metodologia_cientifica.md` seção 4 para o protocolo.

In [1]:
import pandas as pd
from harmonic_signatures.config import PROCESSED_DIR
from harmonic_signatures.disaggregation.dataset import build_disaggregation_dataset
from harmonic_signatures.disaggregation.model import evaluate_disaggregation, per_channel_r2
from harmonic_signatures.features.selection import parse_feature_key, select_features
from harmonic_signatures.schema.models import HARMONIC_FAMILIES, INTERHARMONIC_FAMILIES

harm_matrix = pd.read_parquet(PROCESSED_DIR / 'combined_matrix.parquet')
print(harm_matrix.shape)

(19501, 1297)


## Features combinadas: harmônicas + inter-harmônicas

**Cuidado de nomenclatura**: os sufixos `_harm`/`_inter` no dataset combinado só existem nas colunas onde as duas planilhas têm o mesmo nome (o bloco duplicado H-max investigado na Fase 1) — a grande maioria das colunas genuínas de harmônicas (`H01_Ia_avg`, `THD_Va_avg`...) e inter-harmônicas (`IH00_Ia_avg`...) **não tem sufixo nenhum**. Por isso a categorização usa `parse_feature_key` (família espectral), não o sufixo. Para o bloco H-max duplicado, mantemos apenas a versão `_harm` (fonte canônica) — a versão `_inter` já foi estabelecida como não-confiável (Fase 1).

In [2]:
meta_cols = {'sample_id_harm','sample_id_inter','label','timestamp','timestamp_inter','join_delta_seconds','matched'}
candidate_cols = [c for c in harm_matrix.columns if c not in meta_cols]
specs = {c: parse_feature_key(c) for c in candidate_cols}

harm_cols = [c for c, s in specs.items() if s['family'] in HARMONIC_FAMILIES and s['source_sheet'] != 'interharmonicas']
inter_cols = [c for c, s in specs.items() if s['family'] in INTERHARMONIC_FAMILIES]
feature_cols = harm_cols + inter_cols
print('features harmônicas:', len(harm_cols), '| inter-harmônicas:', len(inter_cols), '| total:', len(feature_cols))

features harmônicas: 666 | inter-harmônicas: 318 | total: 984


## Seleção automática (módulo 8) antes da desagregação

984 candidatas é demais para regressão multi-saída com poucas amostras combinadas (n=4 condições) — reduzir a um conjunto balanceado entre os dois domínios espectrais via ANOVA F-score + remoção de redundância.

In [3]:
sel = select_features(harm_matrix, feature_cols, top_k=60, corr_threshold=0.95, label_col='label')
print('selecionadas:', len(sel.selected_features))
sel.summarize_by('family')

selecionadas: 60


,family,n_selected
0,DC,2
1,H,27
2,IH,27
3,THD3,2
4,THDe,2


## Dataset supervisionado e validação leave-one-condição-out

In [4]:
ds = build_disaggregation_dataset(harm_matrix, sel.selected_features)
print('X shape:', ds.X.shape, '| condições combinadas:', sorted(set(ds.groups)))

report = evaluate_disaggregation(ds)
report

X shape: (6107, 60) | condições combinadas: [np.str_('MÁQUINA BALANCEADA E ALINHADA + (-DIODO 2)'), np.str_('MÁQUINA BALANCEADA E ALINHADA + (-DIODO 3)'), np.str_('MÁQUINA BALANCEADA E ALINHADA + (-DIODO 6)'), np.str_('MÁQUINA BALANCEADA E ALINHADA + RETIFICADOR COMPLETO')]


,held_out_condition,target,mae,r2
0,MÁQUINA BALANCEADA E ALINHADA + (-DIODO 2),machine,0.000000,1.0
1,MÁQUINA BALANCEADA E ALINHADA + (-DIODO 2),source,3.009120,0.0
2,MÁQUINA BALANCEADA E ALINHADA + (-DIODO 3),machine,0.000000,1.0
3,MÁQUINA BALANCEADA E ALINHADA + (-DIODO 3),source,1.203258,0.0
4,MÁQUINA BALANCEADA E ALINHADA + (-DIODO 6),machine,0.000000,1.0
5,MÁQUINA BALANCEADA E ALINHADA + (-DIODO 6),source,1.213624,0.0
6,MÁQUINA BALANCEADA E ALINHADA + RETIFICADOR CO...,machine,0.000000,1.0
7,MÁQUINA BALANCEADA E ALINHADA + RETIFICADOR CO...,source,3.267478,0.0


**Leitura obrigatória**: o alvo `machine` é constante entre as 4 condições (todas usam a mesma máquina-base) — seu R²=1,0 é artefato disso, não prova de separação real. O alvo `source` (o defeito elétrico) é o que de fato testa se o modelo generaliza para uma combinação nunca vista.

## Qualidade de desagregação por canal individual

Quais harmônicos/inter-harmônicos específicos são mais/menos recuperáveis — não só o R² agregado.

In [5]:
per_channel = per_channel_r2(ds, target='source')
print('Top 15 canais melhor desagregados (alvo = defeito elétrico):')
display(per_channel.head(15))
print('15 piores:')
display(per_channel.tail(15))

Top 15 canais melhor desagregados (alvo = defeito elétrico):


,feature,r2,mae
0,H05_Ib_avg,0.214068,4.863387
1,IH20_Ic_avg,0.211070,0.036029
2,IH48_Ic_avg,0.192974,0.078542
3,IH46_Ic_avg,0.189615,0.123774
4,THDe_Ic_avg,0.182020,16.447649
5,H24_Ib_avg,0.147788,0.758702
6,IH49_Ib_avg,0.126402,0.035240
7,H23_Ib_avg,0.118727,0.495024
8,IH01_Ic_avg,0.117717,1.022059
9,H15_Ia_avg,0.024909,0.239469


15 piores:


,feature,r2,mae
45,H03_Ic_avg,-0.844713,4.075367
46,H06_Ic_avg,-0.846682,3.614559
47,IH24_Ia_avg,-0.890766,0.012202
48,THD3_Ic_avg,-0.925152,5.600935
49,THDe_Ib_max,-0.932676,22.579802
50,IH26_Ib_avg,-0.966087,0.050946
51,H05_Va_avg,-1.094469,0.258179
52,H15_Ib_avg,-1.169978,0.248643
53,IH00_Ic_avg,-1.225939,15.333251
54,H39_Ib_avg,-1.424570,0.126134


In [6]:
n_harm_in_top20 = per_channel.head(20)['feature'].isin(harm_cols).sum()
n_inter_in_top20 = per_channel.head(20)['feature'].isin(inter_cols).sum()
print(f'Entre os 20 canais mais bem desagregados: {n_harm_in_top20} harmônicos, {n_inter_in_top20} inter-harmônicos')

Entre os 20 canais mais bem desagregados: 7 harmônicos, 13 inter-harmônicos


## Conclusão preliminar (RQ2)

Preencher após ler os resultados: (1) o R² do alvo `source` melhora incluindo inter-harmônicas frente à versão só-harmônicas (comparar com `notebooks/08_desagregacao_e_predicao.ipynb`); (2) que famílias espectrais dominam os canais mais bem desagregados; (3) lembrar a ressalva de n=4 condições combinadas ao reportar isso como resultado de tese/artigo — tratar como prova de conceito, não validação estatisticamente robusta (ver `docs/metodologia_cientifica.md` seção 9).